In [1]:
!pip install pytorch-lightning

In [2]:
!pip install torchvision

In [3]:
!pip install -U 'tensorboardX'

In [4]:
!pip install -U 'tensorboard'

In [5]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split

from torchvision import transforms
from torchvision.datasets import FashionMNIST

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

import torchmetrics

from typing import Optional, Any, Tuple

# Класс FashionMNISTDataModule

In [6]:
class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = './data', batch_size: int = 128, seed: int = 42):
        super().__init__()
        self.save_hyperparameters()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.seed = seed
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        
        self.train_dataset: Optional[torch.utils.data.Dataset] = None
        self.val_dataset: Optional[torch.utils.data.Dataset] = None
        self.test_dataset: Optional[torch.utils.data.Dataset] = None

    def prepare_data(self):
        FashionMNIST(self.data_dir, train=True, download=True)
        FashionMNIST(self.data_dir, train=False, download=True)

    def setup(self, stage: Optional[str] = None):
        if stage == 'fit' or stage is None:
            full_train = FashionMNIST(self.data_dir, train=True, transform=self.transform)
            
            generator = torch.Generator().manual_seed(self.seed)
            self.train_dataset, self.val_dataset = random_split(
                full_train, [55000, 5000], generator=generator
            )

        if stage == 'test' or stage is None:
            self.test_dataset = FashionMNIST(self.data_dir, train=False, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=4, 
            persistent_workers=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            num_workers=4, 
            persistent_workers=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, 
            batch_size=self.batch_size, 
            num_workers=4, 
            persistent_workers=True
        )

# Класс FashionMNISTModel

По архитектуре ничего осоебенного

У экстраткора признаков просто 2 свертки с ReLU и maxpooling после каждой

Классификация тоже простая, там Flatten->FC->ReLU->Dropout->FC

Решил добавить Dropout с p = 0.2, мне кажется не помешает

In [7]:
class FashionMNISTModel(pl.LightningModule):
    def __init__(self, lr: float = 1e-3, num_classes: int = 10):
        super().__init__()
        self.save_hyperparameters()
        
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

        self.f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="weighted")
        self.roc_auc = torchmetrics.AUROC(task="multiclass", num_classes=num_classes, average="weighted")
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.feature_extractor(x)
        return self.classifier(features)

    def training_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch: Any, batch_idx: int):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        
        preds = torch.softmax(logits, dim=1)
        pred_labels = torch.argmax(logits, dim=1)

        self.f1(pred_labels, y)
        self.roc_auc(preds, y)

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_f1', self.f1, prog_bar=True, on_epoch=True)
        self.log('val_roc_auc', self.roc_auc, on_epoch=True)

    def test_step(self, batch: Any, batch_idx: int):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        
        preds = torch.softmax(logits, dim=1)
        pred_labels = torch.argmax(logits, dim=1)

        self.f1(pred_labels, y)
        self.roc_auc(preds, y)

        self.log('test_loss', loss)
        self.log('test_f1', self.f1)
        self.log('test_roc_auc', self.roc_auc)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), 
            lr=self.hparams.lr, 
            weight_decay=1e-2
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.1, 
            patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

# Гиперпараметры и настройка обучения

По поводу гиперпараметров я выставил `batchsize = 256`. По идее здесь можно сделать нормализацию, так как размер позволяет, но решил не делать. Кол-вл эпох поставил на глаз, в районе 50, так как все равно етсь планировщик, который сам решит когда закончить. lr тоже взял стандартный.

Для планировщик выставил ожидание 10 эпох по `val_loss`. И эпоха сохраняется и подгружается тоже с самым низким `val_loss`. По идее можно взять по `val_f1`, чтобы брать с самой высокой точностью, но я подумал что это странно как то

Критерий остановки или точнее насколько мы допускаем падение лосс `min_delta` я поставил 0.0, то есть любое падение и обновляем лучшую модель

Оптимизатор я решил выбрать AdamW c `weight_decay=1e-2`. Если честно я не знаю какие еще можно тут альтернативы, так как задача не требует специфичного и поэтому стандарт индустрии AdamW выглядит уместно

In [8]:
def train_model(
    random_seed: int = 52, 
    batch_size: int = 256, 
    learning_rate: float = 1e-3, 
    max_epochs: int = 50
) -> Tuple[pl.Trainer, FashionMNISTModel]:
    pl.seed_everything(random_seed, workers=True)
    
    dm = FashionMNISTDataModule(batch_size=batch_size, seed=random_seed)
    model = FashionMNISTModel(lr=learning_rate)
    
    early_stop = EarlyStopping(
        monitor="val_loss", 
        min_delta=0.00, 
        patience=10, 
        verbose=True, 
        mode="min"
    )
    
    checkpoint = ModelCheckpoint(
        monitor="val_loss", 
        mode="min", 
        filename="best-{epoch:02d}"
    )
    
    logger = TensorBoardLogger("tb_logs", name="fashion_mnist_model")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="mps",
        devices=1,
        callbacks=[early_stop, checkpoint],
        logger=logger,
        log_every_n_steps=50,
        num_sanity_val_steps=2,
        deterministic=True
    )

    trainer.fit(model, datamodule=dm)
    trainer.test(datamodule=dm, ckpt_path='best')
    
    return trainer, model

In [9]:
if __name__ == "__main__":
    trainer, model = train_model(random_seed=52)

Seed set to 52
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name              | Type              | Params | Mode  | FLOPs
------------------------------------------------------------------------
0 | feature_extractor | Sequential        | 18.8 K | train | 0    
1 | classifier        | Sequential        | 402 K  | train | 0    
2 | f1                | MulticlassF1Score | 0      | train | 0    
3 | roc_auc           | MulticlassAUROC   | 0      | train | 0    
4 | criterion         | CrossEntropyLoss  | 0      | train | 0    
------------------------------------------------------------------------
421 K     Trainable params
0         Non-trainable params
421 K     Total params
1.687     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0: 100%|██████████| 215/215 [00:03<00:00, 55.14it/s, v_num=17, val_loss=0.415, val_f1=0.838, train_loss=0.588]

Metric val_loss improved. New best score: 0.415


Epoch 1: 100%|██████████| 215/215 [00:02<00:00, 85.93it/s, v_num=17, val_loss=0.329, val_f1=0.883, train_loss=0.359] 

Metric val_loss improved by 0.086 >= min_delta = 0.0. New best score: 0.329


...

Epoch 21: 100%|██████████| 215/215 [00:02<00:00, 79.41it/s, v_num=17, val_loss=0.241, val_f1=0.929, train_loss=0.0688]


Restoring states from the checkpoint path at tb_logs/fashion_mnist_model/version_17/checkpoints/best-epoch=11.ckpt
Loaded model weights from the checkpoint at tb_logs/fashion_mnist_model/version_17/checkpoints/best-epoch=11.ckpt


Testing DataLoader 0: 100%|██████████| 40/40 [00:01<00:00, 23.76it/s]

/Users/iamnoob/Desktop/ML/.venv/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


Testing DataLoader 0: 100%|██████████| 40/40 [00:01<00:00, 21.58it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         test_f1            0.9189741611480713
        test_loss           0.23074311017990112
      test_roc_auc          0.9952820539474487
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Метрики получились очень высокие. Ну это и понятно, задача достаточно простая.
Можно глянуть, что `val_loss` и `test_loss` у нас почти одинаковые, так что какого то переобучения я не вижу. `train_loss` стабильно падал, что тоже хорошо

Обучение завершилось на 21ой эпохе. 
Сохранение и загрузка чекпоинта шла по метрике `val_loss`. С 11 по 21ю эпоху улучшений не было, но не страшно, так как метрики и так крутые